# NDVI - Abril 2026

Procesa imagenes Sentinel-2 L2A de abril 2026 calculando NDVI
con mascara SCL
y exportacion (PNG + TIF + CSV).

**Auto-detecta** si se ejecuta en Google Colab o en PC local.

**ROI**: Shapefile (4ParcelasDefinidas.zip) - 4 parcelas
**Periodo**: 2026-04-01 a 2026-04-30
**Indices**: NDVI
**Filtro nubes**: mascara SCL (clase 4,5) + etiquetado Despejada/Nublada

In [ ]:
# CELDA 1: AUTO-DETECCION DE ENTORNO
import sys, subprocess, os, importlib
EN_COLAB = 'google.colab' in sys.modules
LIBS = ['pystac_client','geopandas','rioxarray','rasterio','odc','xarray','matplotlib']
FALTAN = [l for l in LIBS if not importlib.util.find_spec(l.split('.')[0])]
if EN_COLAB:
    print('Entorno: GOOGLE COLAB')
    if FALTAN:
        get_ipython().system('pip install pystac-client stackstac rioxarray geopandas rasterio odc-stac -q')
else:
    print('Entorno: PC LOCAL')
    if FALTAN:
        print(f'Faltan: {FALTAN}')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install'] + FALTAN + ['-q'])
    else:
        print('Todas las librerias ya instaladas.')
print('Entorno listo.')

In [ ]:
# CELDA 2: IMPORTACIONES
import glob, numpy as np
import pandas as pd, geopandas as gpd
import xarray as xr, rioxarray, rasterio
from datetime import datetime
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pystac_client import Client
from odc.stac import load
from rasterio.features import geometry_mask
if EN_COLAB:
    from google.colab import drive
print('Importaciones completadas.')

In [ ]:
# CELDA 3: CONFIGURACION

if EN_COLAB:
    drive.mount('/content/drive')
    SHAPEFILE_PATH = '/content/drive/MyDrive/Tesis/GEE Murcott/ROI/Parcelas+Definidas.zip'
    BASE_DIR = os.path.join(os.path.dirname(SHAPEFILE_PATH), 'output', 'Imagenes')
else:
    SCRIPT_DIR = os.getcwd()
    SHAPEFILE_PATH = os.path.join(SCRIPT_DIR, '4ParcelasDefinidas.zip')
    if not os.path.exists(SHAPEFILE_PATH):
        SHAPEFILE_PATH = './4ParcelasDefinidas.zip'
    if not os.path.exists(SHAPEFILE_PATH):
        SHAPEFILE_PATH = input('Ruta del shapefile: ').strip()
    BASE_DIR = os.path.join(SCRIPT_DIR, 'output', 'Imagenes')

PERIODO_INICIO = '2026-04-01'
PERIODO_FIN = '2026-04-30'
UMBRAL_NUBES = 10
INDICES_CONFIG = {
    'NDVI': {'cmap': 'RdYlGn', 'vmin': 0.0, 'vmax': 1.0},
}
LISTA_INDICES = list(INDICES_CONFIG.keys())
os.makedirs(BASE_DIR, exist_ok=True)
print('Configuracion cargada.')
print(f'  Periodo: {PERIODO_INICIO} a {PERIODO_FIN}')

In [ ]:
# CELDA 4: FUNCIONES AUXILIARES

def cargar_shapefile(ruta):
    print('Cargando shapefile...')
    gdf = gpd.read_file(ruta)
    print(f'  Features: {len(gdf)}')
    gdf_geo = gdf if (gdf.crs and gdf.crs.is_geographic) else gdf.to_crs('EPSG:4326')
    bbox = gdf_geo.total_bounds
    print(f'  BBox: {bbox}')
    return gdf_geo, bbox

def conectar_stac():
    print('Conectando a Earth Search (AWS)...')
    catalog = Client.open('https://earth-search.aws.element84.com/v1')
    print('  Conexion exitosa.')
    return catalog

def aplicar_mascara_scl(scl, clases_validas=None):
    if clases_validas is None:
        clases_validas = [4, 5]
    mask = xr.zeros_like(scl, dtype=bool)
    for c in clases_validas:
        mask = mask | (scl == c)
    return mask

def aplicar_mascara_geometrica(da, gdf):
    x_name = next((n for n in ['x','lon','longitude'] if n in da.coords), None)
    y_name = next((n for n in ['y','lat','latitude'] if n in da.coords), None)
    if not x_name or not y_name:
        return da
    transform = da.rio.transform()
    ny = da.sizes[y_name]
    nx = da.sizes[x_name]
    mascara = ~geometry_mask(gdf.geometry.values, transform=transform, out_shape=(ny, nx))
    mascara_da = xr.DataArray(mascara, dims=(y_name, x_name),
                               coords={y_name: da[y_name], x_name: da[x_name]})
    return da.where(mascara_da)

def exportar_tif(da, path, crs='EPSG:4326', nodata=-9999):
    try:
        da2 = da.copy()
        x_name = next((n for n in ['x','lon','longitude'] if n in da2.coords), None)
        y_name = next((n for n in ['y','lat','latitude'] if n in da2.coords), None)
        if x_name and y_name:
            da2 = da2.rename({x_name: 'x', y_name: 'y'})
        da2.rio.set_spatial_dims('x', 'y', inplace=True)
        if not da2.rio.crs:
            da2 = da2.rio.write_crs(crs)
        da2.rio.to_raster(path, dtype='float32', compress='lzw', nodata=nodata)
    except Exception:
        data_arr = da.values.astype('float32') if hasattr(da, 'values') else da
        ny, nx = data_arr.shape
        with rasterio.open(path, 'w', driver='GTiff', height=ny, width=nx,
            count=1, dtype='float32', crs=rasterio.crs.CRS.from_string(crs),
            compress='lzw', nodata=nodata) as dst:
            dst.write(data_arr, 1)

def exportar_png(data, path, cmap_name='RdYlGn', vmin=0, vmax=1, dpi=200):
    cmap = plt.get_cmap(cmap_name)
    cmap.set_bad(color='white', alpha=0)
    plt.figure(figsize=(10, 10))
    plt.imshow(np.where(np.isnan(np.asarray(data)), np.nan, np.asarray(data)),
               cmap=cmap, vmin=vmin, vmax=vmax)
    plt.axis('off')
    plt.savefig(path, bbox_inches='tight', pad_inches=0, dpi=dpi)
    plt.close()

def calcular_ndvi(nir, red):
    nir_a = np.asarray(nir).astype('float32')
    red_a = np.asarray(red).astype('float32')
    denom = nir_a + red_a
    ndvi = xr.where(denom == 0, np.nan, (nir_a - red_a) / denom)
    return ndvi.clip(-1, 1)

def calcular_estadisticas_parcela(da, gdf, nombre_indice):
    stats = []
    col_parcela = 'name' if 'name' in gdf.columns else gdf.columns[0]
    for i2, row in gdf.iterrows():
        m = aplicar_mascara_geometrica(da, gdf.iloc[[i2]])
        v = m.values.flatten()
        v = v[~np.isnan(v)]
        stats.append({'ID_Parcela': row.get(col_parcela, str(i2)),
                       f'{nombre_indice}_mean': float(np.mean(v)) if len(v) > 0 else '',
                       f'{nombre_indice}_std': float(np.std(v)) if len(v) > 0 else '',
                       'pixeles_validos': int(len(v))})
    return stats

print('Funciones auxiliares cargadas.')

In [ ]:
# CELDA 5: CARGA SHAPEFILE + BUSQUEDA STAC

gdf_geo, bbox = cargar_shapefile(SHAPEFILE_PATH)
catalog = conectar_stac()

search = catalog.search(
    collections=['sentinel-2-l2a'],
    bbox=list(bbox),
    datetime=f'{PERIODO_INICIO}/{PERIODO_FIN}',
)
items = list(search.items())
if len(items) == 0:
    print('No se encontraron imagenes. Verificar cobertura.')

print(f'Total escenas encontradas: {len(items)}')
for item in items:
    ts = item.datetime
    fecha = ts.strftime('%Y-%m-%d') if ts else 'N/A'
    hora = ts.strftime('%H:%M:%S') if ts else 'N/A'
    cloud = item.properties.get('eo:cloud_cover', -1)
    estado = 'Despejada' if 0 <= cloud < UMBRAL_NUBES else ('Nublada' if cloud >= 0 else 'Sin dato')
    print(f'  {fecha} {hora} | nubes={cloud:.0f}% | {estado}')

In [ ]:
# CELDA 6: PROCESAMIENTO DE ESCENAS

stats_por_indice = {i: [] for i in LISTA_INDICES}
label_mes = 'Abril2026'
contador = 0; exp = 0; omit = 0

for idx, item in enumerate(items):
    ts = item.datetime
    if ts is None: continue
    fecha = ts.strftime('%Y%m%d')
    hora = ts.strftime('%H%M%S')
    cv = item.properties.get('eo:cloud_cover', -1)
    est = 'Despejada' if cv < UMBRAL_NUBES and cv >= 0 else 'Nublada'
    print(f'[{idx+1}/{len(items)}] {fecha} {hora} | {est}')

    # Verificar si el TIF ya existe antes de procesar la escena
    tifs_existentes = 0; tifs_detalle = {}
    for nom in LISTA_INDICES:
        base_nom = f'{nom}_Abril_{contador + 1:03d}_{fecha}_{hora}_{est}'
        tif_path = os.path.join(BASE_DIR, nom, label_mes, 'TIF', f'{base_nom}.tif')
        existe = os.path.exists(tif_path)
        tifs_detalle[nom] = {'existe': existe, 'base': base_nom, 'tif_path': tif_path}
        if existe: tifs_existentes += 1
    if tifs_existentes == len(LISTA_INDICES):
        print('  Todos los TIFs ya existen. Saltando.'); omit += 1; continue
    elif tifs_existentes > 0:
        print(f'  {tifs_existentes}/{len(LISTA_INDICES)} TIFs ya existen. Procesando solo faltantes.')

    try:
        ds = load([item], bands=['red','nir','scl'],
                  bbox=list(bbox), crs='EPSG:4326', resolution=0.0001, groupby=None)
        if ds.sizes.get('time', 0) == 0: continue
        esc = ds.isel(time=0)

        b4 = esc['red'].astype('float32') / 10000
        b8 = esc['nir'].astype('float32') / 10000

        msc = aplicar_mascara_scl(esc['scl'])
        b4 = b4.where(msc); b8 = b8.where(msc)

        b4 = aplicar_mascara_geometrica(b4, gdf_geo)
        b8 = aplicar_mascara_geometrica(b8, gdf_geo)

        ndvi = calcular_ndvi(b8, b4)

        inds = {'NDVI': ndvi}

        contador += 1
        for nom, da in inds.items():
            vals = np.asarray(da.values) if hasattr(da, 'values') else np.asarray(da)
            sf = 'SinDatos' if np.all(np.isnan(vals)) else est
            arch = f'{nom}_Abril_{contador:03d}_{fecha}_{hora}_{sf}'

            d = os.path.join(BASE_DIR, nom, label_mes)
            os.makedirs(f'{d}/TIF', exist_ok=True)
            os.makedirs(f'{d}/PNG', exist_ok=True)

            tif_p = f'{d}/TIF/{arch}.tif'
            if os.path.exists(tif_p): continue

            exportar_tif(da, tif_p)
            cfg = INDICES_CONFIG[nom]
            exportar_png(vals, f'{d}/PNG/{arch}.png',
                         cmap_name=cfg['cmap'], vmin=cfg['vmin'], vmax=cfg['vmax'])

            stats_parcelas = calcular_estadisticas_parcela(da, gdf_geo, nom)
            for st in stats_parcelas:
                stats_por_indice[nom].append({'fecha':fecha,'hora':hora,
                    'id_escena':arch,'nubes_porciento':cv,'estado_nubosidad':sf,
                    **st, 'ruta_tif':tif_p, 'ruta_png':f'{d}/PNG/{arch}.png'})
            exp += 1
        del ds
    except Exception as e:
        print(f'  ERROR: {e}')
        continue

In [ ]:
# CELDA 7: GUARDAR CSV POR INDICE

for nom in LISTA_INDICES:
    reg = stats_por_indice[nom]
    if not reg: print(f'{nom}: sin registros'); continue
    df = pd.DataFrame(reg)
    csv_p = os.path.join(BASE_DIR, nom, label_mes, 'estadisticas.csv')
    df.to_csv(csv_p, index=False, encoding='utf-8')
    print(f'{nom}: {len(df)} registros')

In [ ]:
# CELDA 8: REPORTE FINAL

print('='*60)
print('   REPORTE FINAL')
print('='*60)
print(f'  Escenas: {len(items)} | Exportados: {exp} | Omitidos: {omit}')
print()
tt = 0; tp = 0
for nom in LISTA_INDICES:
    d = os.path.join(BASE_DIR, nom, label_mes)
    nt = len(glob.glob(f'{d}/TIF/*.tif')) if os.path.isdir(f'{d}/TIF') else 0
    np = len(glob.glob(f'{d}/PNG/*.png')) if os.path.isdir(f'{d}/PNG') else 0
    csv_ok = 'CSV' if os.path.exists(f'{d}/estadisticas.csv') else '---'
    tt += nt; tp += np
    print(f'  {nom:15s}: {nt:3d} TIFs | {np:3d} PNGs | {csv_ok}')
print()
print(f'  TOTAL: {tt} TIFs + {tp} PNGs')
print(f'  Directorio: {BASE_DIR}')
print('='*60)
print('  Procesamiento completado!')
print('='*60)